[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C32_Skills_Tools_Course/01_skill_loading/01_skill_loading.ipynb)

# 01 · Skill 定义与加载

目标：用**纯 Python 标准库**（不依赖任何框架, 连 PyYAML 都不用）从零写出一个 skill 加载器——**frontmatter 解析 → 发现并建注册表 → 相关性触发 → 按需加载正文 → 注入上下文 → token 预算取舍**，全程 `assert` 验证、**无需 API key**。

路线：手写 frontmatter 解析器 → skill 发现/注册表 → 相关性触发(关键词 + MockLLM) → 按需加载+注入(渐进披露) → token 预算控制器 → ✏️ 练习 → 📖 答案 → 🧪 真实 SKILL.md 胶囊。

> 心智模型：**skill = 把「这类任务怎么做」写成独立文件, 平时只露一行简介, 用到才把全文请进上下文**。难点在简介前面的解析、发现、选择、注入。

## 1 · 从零写 frontmatter 解析器（纯标准库）

一份 skill 是 `---\nkey: value\n...\n---\n<markdown 正文>`。我们**不依赖 PyYAML**, 手写一个极简解析器：

- 切出 frontmatter 块与正文；
- 解析 frontmatter 的 `key: value`（顶层、字符串值, 值里允许含冒号）；
- 对残缺输入（无 frontmatter / 缺结尾 `---`）有明确行为。

这正是「不依赖框架」的第一课：连解析都自己掌控, 才看得清边界。

In [ ]:
import json, os, textwrap

def split_frontmatter(text):
    '''把 SKILL.md 文本切成 (frontmatter_dict, body_str)。
       约定: 文件以一行 --- 开头, 到下一行 --- 结束, 中间是 key: value。
       无合法 frontmatter -> ({}, 整段文本)。残缺(缺结尾---) -> 视为无 frontmatter。'''
    lines = text.split('\n')
    if not lines or lines[0].strip() != '---':
        return {}, text                      # 无 frontmatter
    # 找结尾的 ---
    end = None
    for i in range(1, len(lines)):
        if lines[i].strip() == '---':
            end = i
            break
    if end is None:
        return {}, text                      # 残缺: 没有结尾 ---, 当作无 frontmatter
    fm = {}
    for line in lines[1:end]:
        if not line.strip() or line.lstrip().startswith('#'):
            continue                          # 跳过空行与注释
        if ':' not in line:
            continue                          # 不是 key: value, 跳过
        key, val = line.split(':', 1)         # 只切第一个冒号, 值里允许再有冒号
        fm[key.strip()] = val.strip()
    body = '\n'.join(lines[end + 1:]).lstrip('\n')
    return fm, body

SAMPLE = '''---
name: git-commit
description: 撰写规范的 git 提交信息。当用户要提交代码时使用。
---
# 写 Git 提交信息
- 首行祈使句, 50 字以内, 不加句号。
- 格式: feat/fix/docs: 简述'''

fm, body = split_frontmatter(SAMPLE)
print('frontmatter:', fm)
print('正文前 20 字:', body[:20])
assert fm['name'] == 'git-commit'
assert '当用户要提交代码时使用' in fm['description']   # 值里含冒号也解析正确
assert body.startswith('# 写 Git 提交信息')           # 正文不含 frontmatter
# 边界: 无 frontmatter / 残缺 frontmatter
assert split_frontmatter('# 只是普通 markdown') == ({}, '# 只是普通 markdown')
assert split_frontmatter('---\nname: x\n# 没有结尾')[0] == {}   # 残缺 -> 当无
print('✅ frontmatter 解析器：切块正确、含冒号的值正确、残缺输入不崩溃')

## 2 · 校验一份 skill：必备字段与清晰报错

解析只是第一步。一份合法 skill 必须有 `name` 和 `description`（发现与触发都靠它们）。

校验失败要给**清晰报错**（哪份 skill、缺什么）, 而不是稍后在某个 `KeyError` 上莫名崩溃——这是健壮加载器的基本功。

In [ ]:
REQUIRED_FIELDS = ('name', 'description')

def validate_skill(fm, source='<unknown>'):
    '''校验 frontmatter 是否含必备字段。返回 (ok, 错误信息或None)。'''
    if not fm:
        return False, f'{source}: 缺少 frontmatter(--- 元数据块)'
    for field in REQUIRED_FIELDS:
        if field not in fm or not fm[field].strip():
            return False, f'{source}: 缺少必备字段 {field}'
    return True, None

ok, err = validate_skill(fm, 'git-commit.md')
print('合法 skill ->', ok, err)
assert ok is True and err is None
# 残缺: 无 frontmatter
ok2, err2 = validate_skill({}, 'broken.md')
print('无 frontmatter ->', ok2, err2)
assert ok2 is False and 'frontmatter' in err2
# 残缺: 缺 description
ok3, err3 = validate_skill({'name': 'x'}, 'no-desc.md')
print('缺 description ->', ok3, err3)
assert ok3 is False and 'description' in err3
print('✅ skill 校验：必备字段齐全才放行, 失败信息点名「哪份 skill 缺什么」')

## 3 · 发现与注册表：只读 frontmatter, 正文留待按需加载

发现 = 扫描一批 skill 源、解析每份的 frontmatter、建一张**轻量注册表**（`name -> 描述 + 正文路径/来源`）。

**纪律（渐进披露的落地）：发现阶段只解析 frontmatter, 不读正文**。这里用一个内存字典模拟「磁盘」, 但「描述常驻、正文惰性取」的结构与真实一致。命名冲突要有明确策略——本课用最严的「报错」。

In [ ]:
# 用一个 dict 模拟磁盘: source_name -> 文件全文(含 frontmatter + 正文)
SKILL_FILES = {
    'git.md': '---\nname: git\ndescription: 处理 git 提交、分支、合并等版本控制任务。\n---\n# git\n首行祈使句, 50 字内。',
    'sql.md': '---\nname: sql\ndescription: 编写和优化 SQL 查询, 处理数据库相关任务。\n---\n# sql\n避免 SELECT *, 加合适索引。',
    'docs.md': '---\nname: docs\ndescription: 撰写和润色文档、说明、注释。\n---\n# docs\n先列大纲, 每段一个要点。',
}

class SkillRegistry:
    '''轻量注册表: 只存描述 + 来源(正文路径), 不存正文本身。'''
    def __init__(self):
        self.skills = {}     # name -> {'description', 'source'}
        self._files = {}     # source -> 文件全文(模拟磁盘: 真实系统这里是路径, 正文留在盘上)
    def discover(self, files):
        self._files = dict(files)                 # 记住「磁盘」, load_body 时再回来读
        for source, text in files.items():
            fm, _body = split_frontmatter(text)   # 只取 frontmatter, 丢弃正文(惰性!)
            ok, err = validate_skill(fm, source)
            if not ok:
                raise ValueError(err)
            name = fm['name']
            if name in self.skills:               # 命名冲突 -> 报错(最严策略)
                raise ValueError(f'skill 重名: {name} (来自 {source} 与已注册)')
            self.skills[name] = {'description': fm['description'], 'source': source}
        return self
    def catalog(self):
        '''给模型看的轻量目录: name + 一句话描述(常驻上下文)。'''
        return {n: meta['description'] for n, meta in self.skills.items()}
    def load_body(self, name):
        '''此刻才从「磁盘」读正文(渐进披露的兑现)。'''
        source = self.skills[name]['source']
        _fm, body = split_frontmatter(self._files[source])
        return body

reg = SkillRegistry().discover(SKILL_FILES)
print('已发现 skill:', list(reg.skills))
print('轻量目录:', reg.catalog())
assert set(reg.skills) == {'git', 'sql', 'docs'}
# 注册表里没有正文(只有描述+来源) —— 证明发现没有加载正文
assert 'description' in reg.skills['git'] and 'body' not in reg.skills['git']
assert reg.load_body('git').startswith('# git')   # 需要时才取得到正文
print('✅ 发现: 只解析 frontmatter 建轻量目录, 正文留待 load_body 按需取')

## 4 · 相关性触发：关键词版 + MockLLM 版

选择 = 给定任务, 挑出相关 skill 的 name。两种实现:

- **关键词版**: 任务文本命中 description 里的关键词（确定、可断言）；
- **MockLLM 版**: 把**轻量目录**（描述清单, 这正是描述必须常驻的原因）连同任务发给模型, 让它返回相关 name。

两者下游（加载+注入）完全一致——所以可无缝换成真实模型。

In [ ]:
class MockLLM:
    '''确定性假模型: 按规则把 prompt 映射到响应。第一个命中的规则生效。
       这里用它模拟「模型读目录、返回相关 skill 名列表」。'''
    def __init__(self, rules, default=None):
        self.rules = rules
        self.default = default if default is not None else {'type': 'text', 'text': '[]'}
        self.calls = 0
    def __call__(self, prompt):
        self.calls += 1
        text = prompt if isinstance(prompt, str) else json.dumps(prompt, ensure_ascii=False)
        for kw, resp in self.rules:
            if kw in text:
                return json.loads(json.dumps(resp))
        return json.loads(json.dumps(self.default))

def select_by_keyword(task, registry):
    '''朴素相关性: 任务里出现 description 的某个词(去标点切词)。'''
    hits = []
    for name, desc in registry.catalog().items():
        words = desc.replace('、', ' ').replace(',', ' ').replace('。', ' ').split()
        if any(w and w in task for w in words):
            hits.append(name)
    return hits

def select_by_llm(task, registry, llm):
    '''真实路径模拟: 把轻量目录+任务发给模型, 模型返回相关 name 的 JSON 列表。'''
    catalog = registry.catalog()
    prompt = f'目录: {json.dumps(catalog, ensure_ascii=False)}\n任务: {task}\n返回相关 skill 名的 JSON 列表。'
    resp = llm(prompt)
    names = json.loads(resp['text'])
    return [n for n in names if n in registry.skills]   # 过滤掉幻觉名

task = '帮我写一个 git 提交信息'
kw_hits = select_by_keyword(task, reg)
print('关键词触发:', kw_hits)
assert 'git' in kw_hits and 'sql' not in kw_hits

# MockLLM 版: 规则模拟「看到含 提交/git 的任务就选 git」
llm = MockLLM(rules=[('提交', {'type': 'text', 'text': '["git"]'})])
llm_hits = select_by_llm(task, reg, llm)
print('MockLLM 触发:', llm_hits)
assert llm_hits == ['git']
assert llm.calls == 1
print('✅ 相关性触发: 关键词版与 MockLLM 版都只选中 git; 下游加载/注入两者通用')

## 5 · 按需加载 + 注入：渐进披露的兑现

选出相关 name 后, 才从「磁盘」加载这几份（**且只有这几份**）的正文, 拼成注入上下文。

**核心不变量: 未命中的 skill, 其正文绝不出现在最终上下文里**——这是渐进披露是否真生效的试金石。

In [ ]:
def load_and_inject(names, registry):
    '''按需加载命中 skill 的正文并拼成注入上下文(带清晰边界标记)。'''
    chunks = []
    for n in names:
        body = registry.load_body(n)          # 此刻才读正文(惰性)
        chunks.append(f'## skill: {n}\n{body}')
    return '\n\n'.join(chunks)

task = '帮我写一个 git 提交信息'
hits = select_by_keyword(task, reg)
context = load_and_inject(hits, reg)
print('注入上下文:\n', context)
# 命中的 git 正文在
assert '## skill: git' in context and '首行祈使句' in context
# 渐进披露试金石: 未命中的 sql/docs 正文绝不在上下文里
assert '避免 SELECT' not in context, 'sql 未命中, 其正文不该被注入!'
assert '先列大纲' not in context, 'docs 未命中, 其正文不该被注入!'
# 量化对比: 全量加载 vs 按需加载
full = '\n\n'.join(reg.load_body(n) for n in reg.skills)
print(f'全量加载 {len(full)} 字 vs 按需加载 {len(context)} 字')
assert len(context) < len(full), '渐进披露应当显著更省上下文'
print('✅ 按需加载+注入: 只有命中的正文进上下文, 未命中的一字不进 —— 渐进披露生效')

## 6 · token 预算控制器：多份命中时的取舍

若一个任务命中多份 skill, 正文加起来可能超预算。预算控制器: **按相关性得分降序、贪心装入, 超预算则停**。

本课用**字符数近似 token**（真实系统用 tokenizer）。不变量: 注入总量 ≤ 预算, 且优先保留高相关。

In [ ]:
def budget_select(scored, registry, budget):
    '''scored = [(name, 相关性得分), ...]; 按得分降序贪心装入, 累计正文字符数不超 budget。
       返回 (选中的 name 列表, 累计字符数)。'''
    ordered = sorted(scored, key=lambda x: x[1], reverse=True)
    chosen, used = [], 0
    for name, _score in ordered:
        size = len(registry.load_body(name))     # 近似 token 成本
        if used + size > budget:
            continue                              # 这份装不下, 跳过(低分先被挤掉)
        chosen.append(name)
        used += size
    return chosen, used

# 三份都命中, 但预算只够装下其中相关性最高的若干份
scored = [('git', 0.9), ('sql', 0.5), ('docs', 0.3)]
sizes = {n: len(reg.load_body(n)) for n in reg.skills}
print('各 skill 正文字符数:', sizes)
budget = sizes['git'] + 1          # 预算只够装 git 一份
chosen, used = budget_select(scored, reg, budget)
print(f'预算 {budget}, 选中 {chosen}, 用了 {used}')
assert chosen == ['git'], '预算只够最高相关的 git'
assert used <= budget
# 放宽预算到能装 git+sql 但装不下 docs
budget2 = sizes['git'] + sizes['sql'] + 1
chosen2, used2 = budget_select(scored, reg, budget2)
assert chosen2 == ['git', 'sql'] and 'docs' not in chosen2
assert used2 <= budget2
print('✅ token 预算: 按相关性降序贪心装入, 总量不超预算, 低相关被挤掉')

---
## ✏️ 练习 1：frontmatter 解析——支持简单列表值

上面的解析器只处理 `key: value` 标量。真实 frontmatter 常有**内联列表**, 如 `tags: [git, vcs]`。

实现 `parse_inline_list(value)`: 若 value 形如 `[a, b, c]` 返回列表 `['a','b','c']`（去空白）；否则原样返回字符串。

再实现 `split_frontmatter_v2(text)`: 复用 `split_frontmatter`, 但把形如 `[...]` 的值解析成列表。

In [ ]:
def parse_inline_list(value):
    # TODO: 若 value.strip() 以 [ 开头、] 结尾, 去掉方括号, 按逗号切分、去空白, 返回列表;
    #       否则返回原 value 字符串。空列表 '[]' 应返回 []。
    raise NotImplementedError

def split_frontmatter_v2(text):
    # TODO: 调 split_frontmatter 得 (fm, body); 对 fm 每个值调 parse_inline_list; 返回 (新fm, body)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert parse_inline_list('[git, vcs, scm]') == ['git', 'vcs', 'scm']
assert parse_inline_list('[]') == []
assert parse_inline_list('普通字符串') == '普通字符串'
TXT = '---\nname: git\ntags: [git, vcs]\ndescription: 版本控制\n---\n正文'
fm2, body2 = split_frontmatter_v2(TXT)
assert fm2['tags'] == ['git', 'vcs']      # 列表值
assert fm2['name'] == 'git'               # 标量值不变
assert body2 == '正文'
print('✅ 练习 1 通过: 内联列表值被正确解析, 标量值不受影响')

## ✏️ 练习 2：相关性触发——返回带得分的排序结果

把关键词触发升级为**带得分**: 一份 skill 的得分 = 它的 description 关键词在任务里命中的**个数**。

实现 `score_skills(task, registry)`: 返回 `[(name, 命中词数), ...]`, **只含得分>0 的, 且按得分降序**。

In [ ]:
def score_skills(task, registry):
    # TODO: 对每份 skill, 把 description 用 、, 。 切词, 统计有多少个词出现在 task 里;
    #       命中数>0 的收进结果; 最后按命中数降序返回 [(name, count), ...]
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 构造一个 description 多词命中的任务
task = '帮我优化这条 SQL 查询, 它在数据库里很慢'
scored = score_skills(task, reg)
print('得分:', scored)
names = [n for n, _ in scored]
assert 'sql' in names                     # sql 的 SQL/查询/数据库 多词命中
assert all(s > 0 for _, s in scored)      # 只含命中的
assert scored == sorted(scored, key=lambda x: x[1], reverse=True)  # 降序
assert scored[0][0] == 'sql'              # sql 命中词最多, 排第一
print('✅ 练习 2 通过: 带得分的相关性、只含命中、按得分降序')

## ✏️ 练习 3：完整加载流水线 + 预算

把发现→评分→预算→加载注入**串成一条流水线**。

实现 `load_skills(task, registry, budget)`: ① 用 `score_skills` 评分(练习 2)；② 用 `budget_select` 在预算内选取(已给)；③ 加载注入选中的正文。返回 `(注入上下文, 选中的 name 列表)`。未命中任何 skill 时返回 `('', [])`。

In [ ]:
def load_skills(task, registry, budget=10000):
    # TODO: scored = score_skills(task, registry)
    #       若 scored 为空, 返回 ('', [])
    #       chosen, _ = budget_select(scored, registry, budget)
    #       context = load_and_inject(chosen, registry)
    #       返回 (context, chosen)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
ctx, chosen = load_skills('帮我写 git 提交信息', reg, budget=10000)
assert chosen == ['git']
assert '## skill: git' in ctx and '避免 SELECT' not in ctx   # 渐进披露仍成立
# 无关任务: 不命中任何 skill
ctx0, chosen0 = load_skills('今天天气真好啊随便聊聊', reg, budget=10000)
assert chosen0 == [] and ctx0 == ''
# 预算卡死: 即使命中也装不下
ctx1, chosen1 = load_skills('帮我写 git 提交信息', reg, budget=1)
assert chosen1 == [] and ctx1 == ''
print('✅ 练习 3 通过: 发现→评分→预算→注入 全流水线, 边界(无命中/预算不足)都正确')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def parse_inline_list(value):
    v = value.strip()
    if v.startswith('[') and v.endswith(']'):
        inner = v[1:-1].strip()
        if not inner:
            return []
        return [x.strip() for x in inner.split(',')]
    return value

def split_frontmatter_v2(text):
    fm, body = split_frontmatter(text)
    fm2 = {k: parse_inline_list(v) for k, v in fm.items()}
    return fm2, body

In [ ]:
# 练习 2 参考答案
def score_skills(task, registry):
    out = []
    for name, desc in registry.catalog().items():
        words = desc.replace('、', ' ').replace(',', ' ').replace('。', ' ').split()
        count = sum(1 for w in words if w and w in task)
        if count > 0:
            out.append((name, count))
    out.sort(key=lambda x: x[1], reverse=True)
    return out

In [ ]:
# 练习 3 参考答案
def load_skills(task, registry, budget=10000):
    scored = score_skills(task, registry)
    if not scored:
        return '', []
    chosen, _ = budget_select(scored, registry, budget)
    context = load_and_inject(chosen, registry)
    return context, chosen

---
## 🧪 真实数据胶囊：真实 SKILL.md 的形状

下面是几份**贴近真实**的 SKILL.md（形如 Anthropic Agent Skills / Claude Code `.claude/skills/<name>/SKILL.md`）。我们用上面写的解析器去解析它们, 体会真实 skill 与本课实现的逐行对应。

> 真实形状: `---` frontmatter(`name`/`description`, 可能还有 `allowed-tools` 等) + markdown 正文(详细指令、示例、注意事项)。

In [ ]:
# 真实风格的 SKILL.md(贴近 Anthropic Agent Skills 文档样例)
REAL_SKILLS = {
    'pdf-fill/SKILL.md': '''---
name: pdf-form-filler
description: 填写 PDF 表单。当用户提供 PDF 表单并要求填入数据时使用。
---
# 填写 PDF 表单
1. 用 pypdf 读取表单字段名。
2. 把用户数据按字段名映射。
3. 写回并保存为新文件, 不覆盖原件。''',
    'commit/SKILL.md': '''---
name: git-commit
description: 撰写规范的 git 提交信息。提交代码、写 commit message 时使用。
---
# Git 提交信息规范
- 首行: <type>: <subject>, 祈使句, <=50 字。
- type 取值: feat, fix, docs, refactor, test, chore。''',
}

real_reg = SkillRegistry().discover(REAL_SKILLS)
print('解析出的真实 skill:', list(real_reg.skills))
print('pdf 的描述:', real_reg.skills['pdf-form-filler']['description'])
assert set(real_reg.skills) == {'pdf-form-filler', 'git-commit'}
# 正文仍是惰性的: 注册表只有描述
body = real_reg.load_body('git-commit')
assert 'feat' in body and 'fix' in body
# 本课的解析器直接吃真实 SKILL.md
fm_real, _ = split_frontmatter(REAL_SKILLS['pdf-fill/SKILL.md'])
assert fm_real['name'] == 'pdf-form-filler'
print('✅ 本课解析器/注册表直接适用于真实 SKILL.md(frontmatter + 正文)')

**🧪 胶囊练习**：实现 `catalog_size(registry)`：返回**轻量目录**（所有 description 拼起来）的总字符数——这是「常驻上下文成本」的近似。真实系统据此判断「目录本身会不会太大、要不要先检索粗筛」。

In [ ]:
def catalog_size(registry):
    # TODO: 返回 sum(len(desc) for desc in registry.catalog().values())
    raise NotImplementedError

In [ ]:
# 自测
size = catalog_size(real_reg)
expected = sum(len(d) for d in real_reg.catalog().values())
assert size == expected and size > 0
print('轻量目录常驻成本(字符):', size)
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def catalog_size(registry):
    return sum(len(desc) for desc in registry.catalog().values())

---
## 🔧 旁注：对应的真实 Claude 调用长什么样（无 key 自动回退 MockLLM）

本课用 MockLLM 模拟的「相关性触发」, 换成真实 Claude 只是把**选择 skill**那一步交给模型——其余（发现、加载、注入、预算）**一字不改**。

按全课统一范式: **有 `ANTHROPIC_API_KEY` 走真实 `messages.create`, 没有就自动回退 MockLLM**, 代码永远跑得通（本环境无 key, 故走 MockLLM）。

In [ ]:
def make_llm(rules=None, default=None, model='claude-sonnet-4-6'):
    '''统一 LLM 工厂: 有 ANTHROPIC_API_KEY -> 真实 Claude; 否则 -> MockLLM。'''
    if os.environ.get('ANTHROPIC_API_KEY'):
        try:
            import anthropic
            client = anthropic.Anthropic()
            def real_llm(prompt):
                msgs = prompt if isinstance(prompt, list) else [{'role': 'user', 'content': prompt}]
                resp = client.messages.create(model=model, max_tokens=1024, messages=msgs)
                txt = ''.join(b.text for b in resp.content if b.type == 'text')
                return {'type': 'text', 'text': txt}
            print(f'[make_llm] 使用真实 Claude: {model}')
            return real_llm
        except Exception as e:
            print(f'[make_llm] 真实 Claude 不可用({type(e).__name__}), 回退 MockLLM')
    print('[make_llm] 无 API key, 使用 MockLLM')
    return MockLLM(rules or [], default=default)

# 用统一工厂跑相关性触发: scaffold(select_by_llm)不变, 只换 llm 来源
llm2 = make_llm(rules=[('提交', {'type': 'text', 'text': '["git"]'})])
hits = select_by_llm('帮我写 git 提交信息', reg, llm2)
assert hits == ['git']
print('✅ make_llm 就位: 无 key 回退 MockLLM, 有 key 走真实 Claude; select_by_llm 逻辑不变')

> 在真实 Claude Code 里, skill 的 description 清单会随 system 一起常驻, 模型在每轮自行判断该激活哪些 skill、由运行时注入正文。你在本课写的 `split_frontmatter` / `SkillRegistry` / `load_and_inject` / `budget_select` **原样适用**——这就是「零件可迁移」的含义。

### 小结
- **skill = frontmatter(name/description) + 正文**: 平时只露描述, 用到才加载正文。
- **渐进披露(贯穿全课)**: 发现只读描述(轻、可全量), 命中才加载正文(重、必精选)——上下文成本 = Σ描述 + Σ命中正文。
- **发现**: 扫描+解析 frontmatter 建轻量注册表, 命名冲突要有明确策略, 残缺 frontmatter 不崩溃。
- **相关性触发**: 关键词版可断言、MockLLM 版贴真实, 下游加载/注入两者通用。
- **注入试金石**: 未命中的正文绝不进上下文。**token 预算**: 按相关性降序贪心装入、总量不超预算。

下一站：**模块 02 · Slash 命令与动态注入** —— 另一条能力入口：用户显式触发的命令, 带参数绑定与 `!`shell``/`@file` 动态注入。